<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Key insights:

RAG improves context. Agents take action like calling tools, affecting the world beyond the conversation.Think → call a tool → observe the result → repeat. You made this concrete by inspecting the chat history and watching each cycle unfold. Tools are just Python functions. The model reads the docstring to know when and how to use each one. Multi-tool reasoning when one query triggered multiple tool calls. The agent decided the order on its own, no hardcoding needed. Adding get_pe_ratio showed you're always in control of what the agent can do. A P/E above ~25 signals potential overvaluation, the tool turned a number into an opinion. Instead of pre-defining tools, the agent writes its own Python on the fly. Far more flexible for open-ended analysis.


In [ ]:
# Setup
!pip install -q -U google-generativeai
!pip install -q yfinance

import google.generativeai as genai
from google.colab import userdata
import os
import yfinance as yf

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Original Tools (unchanged)

def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"


def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... ⚠️ TOOL CALL: Fetching Risk Metrics for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)

        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"

        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"

In [ ]:
# NEW Tool: get_pe_ratio

def get_pe_ratio(ticker: str):
    """
    Fetches the trailing Price-to-Earnings (P/E) ratio for a given stock.
    A P/E ratio above the market average (~25) may indicate the stock
    is overvalued relative to the broader market.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT').
    """
    print(f"  ... 📊 TOOL CALL: Fetching P/E Ratio for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', 'N/A')

        if pe == 'N/A' or pe is None:
            return {"ticker": ticker, "pe_ratio": "N/A", "verdict": "P/E data unavailable"}

        pe = round(pe, 2)

        # Compare against the commonly used market average P/E of 25
        market_average_pe = 25
        if pe > market_average_pe:
            verdict = f"Potentially OVERVALUED (P/E {pe} > market average of {market_average_pe})"
        else:
            verdict = f"Potentially FAIRLY VALUED or UNDERVALUED (P/E {pe} <= market average of {market_average_pe})"

        return {
            "ticker": ticker,
            "pe_ratio": pe,
            "market_average_pe": market_average_pe,
            "verdict": verdict
        }
    except Exception as e:
        return f"Error fetching P/E ratio for {ticker}: {e}"


print("✅ All three tools defined (including new get_pe_ratio).")

✅ All three tools defined (including new get_pe_ratio).


In [ ]:
# 4. Initialise Model with All Three Tools

tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]  # <-- get_pe_ratio added

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Model initialised with 3 agentic tools.")

✅ Model initialised with 3 agentic tools.


In [ ]:
# 5. Task 16 Query — Is Apple Overvalued?

query = (
    "Using the P/E ratio tool, find Apple's current P/E ratio. "
    "The average market P/E is 25. Based on this, is Apple overvalued, "
    "fairly valued, or undervalued? Also tell me its current stock price."
)

print(f"\n❓ USER QUERY:\n{query}\n")

response = chat.send_message(query)

print("\n🤖 AGENT RESPONSE:")
print(response.text)


❓ USER QUERY:
Using the P/E ratio tool, find Apple's current P/E ratio. The average market P/E is 25. Based on this, is Apple overvalued, fairly valued, or undervalued? Also tell me its current stock price.

  ... 📊 TOOL CALL: Fetching P/E Ratio for AAPL ...
  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for AAPL ...

🤖 AGENT RESPONSE:
Apple's current P/E ratio is 31.35. Compared to the average market P/E of 25, Apple appears to be potentially overvalued. Apple's current stock price is $247.99.


In [ ]:
# 6. History Inspection

print("\n" + "="*55)
print("🕵️  HISTORY INSPECTION — Agent's Thought Process")
print("="*55 + "\n")

for message in chat.history:
    role = message.role
    print(f"--- {role.upper()} ---")
    for part in message.parts:
        if fn := part.function_call:
            print(f"🔧 ACTION: Calling '{fn.name}' with args: {dict(fn.args)}")
        elif resp := part.function_response:
            print(f"📨 OBSERVATION: Tool returned: {resp.response}")
        else:
            print(part.text)
    print()


🕵️  HISTORY INSPECTION — Agent's Thought Process

--- USER ---
Using the P/E ratio tool, find Apple's current P/E ratio. The average market P/E is 25. Based on this, is Apple overvalued, fairly valued, or undervalued? Also tell me its current stock price.

--- MODEL ---
🔧 ACTION: Calling 'get_pe_ratio' with args: {'ticker': 'AAPL'}
🔧 ACTION: Calling 'get_stock_price' with args: {'ticker': 'AAPL'}

--- USER ---
📨 OBSERVATION: Tool returned: <proto.marshal.collections.maps.MapComposite object at 0x78145e379c10>
📨 OBSERVATION: Tool returned: <proto.marshal.collections.maps.MapComposite object at 0x78145e37a570>

--- MODEL ---
Apple's current P/E ratio is 31.35. Compared to the average market P/E of 25, Apple appears to be potentially overvalued. Apple's current stock price is $247.99.

